# Synthetic Requests Generator

Этот notebook генерирует синтетические пользовательские запросы на основе реальных запросов.

Пайплайн состоит из нескольких LLM-запросов:

1. Анализ реальных запросов и построение таксономии.
2. Генерация synthetic requests по реальным примерам.
3. LLM-валидация synthetic requests.
4. Локальная валидация: дубли, похожесть на реальные запросы, placeholders, PII-like паттерны.
5. Repair-запрос для исправления невалидных synthetic requests.
6. Сохранение результата в `.jsonl`.

Перед запуском укажи `OPENAI_API_KEY` в переменных окружения или в отдельной защищённой ячейке.


In [ ]:
# Если OpenAI SDK и Pydantic не установлены, раскомментируй:
# %pip install openai pydantic


In [ ]:
from __future__ import annotations

import difflib
import hashlib
import json
import os
import random
import re
import time
from pathlib import Path
from typing import Any

from openai import OpenAI
from pydantic import BaseModel, Field, field_validator


## 1. Схемы данных

Используем Pydantic-схемы, чтобы модель возвращала структурированный результат, а не произвольный текст.


In [ ]:
class RealRequest(BaseModel):
    id: str
    text: str
    metadata: dict[str, Any] = Field(default_factory=dict)

    @field_validator("text")
    @classmethod
    def non_empty_text(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("Real request text cannot be empty")
        return value


class DatasetTaxonomy(BaseModel):
    domains: list[str] = Field(description="Main domains/topics found in real requests.")
    intents: list[str] = Field(description="User intents found in real requests.")
    complexity_levels: list[str] = Field(description="Observed complexity levels.")
    style_patterns: list[str] = Field(description="Observed language/style/noise patterns.")
    constraints_patterns: list[str] = Field(description="Common constraint types.")
    forbidden_copy_patterns: list[str] = Field(description="Patterns that must not be copied into synthetic data.")

    @field_validator(
        "domains",
        "intents",
        "complexity_levels",
        "style_patterns",
        "constraints_patterns",
        "forbidden_copy_patterns",
    )
    @classmethod
    def non_empty_list(cls, value: list[str]) -> list[str]:
        cleaned = [x.strip() for x in value if x.strip()]
        if not cleaned:
            raise ValueError("List cannot be empty")
        return cleaned


class SyntheticRequest(BaseModel):
    synthetic_id: str = Field(description="Stable unique id, e.g. synth_000001.")
    user_request: str = Field(description="The generated synthetic user request.")
    domain: str = Field(description="Domain/topic.")
    intent: str = Field(description="User intent.")
    complexity: str = Field(description="simple, medium, complex, or expert.")
    expected_answer_type: str = Field(description="Expected assistant answer type.")
    source_example_ids: list[str] = Field(description="IDs of real examples that inspired this request.")
    variation_strategy: str = Field(description="How this synthetic request differs from real examples.")
    quality_notes: str = Field(description="Why this synthetic request is realistic and useful.")

    @field_validator("user_request")
    @classmethod
    def request_must_be_non_empty(cls, value: str) -> str:
        value = value.strip()
        if len(value) < 12:
            raise ValueError("Synthetic request is too short")
        return value

    @field_validator("source_example_ids")
    @classmethod
    def source_ids_required(cls, value: list[str]) -> list[str]:
        if not value:
            raise ValueError("source_example_ids cannot be empty")
        return value


class SyntheticBatch(BaseModel):
    items: list[SyntheticRequest]


class ValidationItem(BaseModel):
    synthetic_id: str
    is_valid: bool
    score: int = Field(ge=0, le=100)
    issues: list[str]
    suggested_fix: str


class ValidationReport(BaseModel):
    items: list[ValidationItem]
    batch_score: int = Field(ge=0, le=100)
    global_issues: list[str]


## 2. Конфиг и локальные проверки


In [ ]:
DEFAULT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

MIN_REQUEST_LEN = 12
MAX_SIMILARITY_TO_REAL = 0.88
MAX_SIMILARITY_TO_ACCEPTED = 0.92
MIN_LLM_VALIDATION_SCORE = 75

BAD_PLACEHOLDERS = [
    "lorem ipsum",
    "todo",
    "xxx",
    "пример запроса",
    "как в примере",
    "[вставьте",
    "<insert",
    "{placeholder}",
]

PII_PATTERNS = [
    r"\b[\w\.-]+@[\w\.-]+\.\w+\b",       # email
    r"\+?\d[\d\s\-\(\)]{8,}\d",          # phone-ish
    r"\b\d{4}\s?\d{4}\s?\d{4}\s?\d{4}\b" # card-ish
]


def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\wа-яёіїєґ\s\?\!\.,:;/-]", "", text, flags=re.IGNORECASE)
    return text


def similarity(a: str, b: str) -> float:
    return difflib.SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()


def stable_hash(text: str) -> str:
    return hashlib.sha1(normalize_text(text).encode("utf-8")).hexdigest()[:12]


def contains_bad_placeholder(text: str) -> bool:
    low = text.lower()
    return any(x in low for x in BAD_PLACEHOLDERS)


def contains_pii_like_text(text: str) -> bool:
    return any(re.search(pattern, text) for pattern in PII_PATTERNS)


def examples_block(real_requests: list[RealRequest], max_examples: int = 30) -> str:
    sample = real_requests[:max_examples]
    lines = []
    for item in sample:
        meta = ""
        if item.metadata:
            meta = f" | metadata={json.dumps(item.metadata, ensure_ascii=False)}"
        lines.append(f"- id={item.id}{meta}\n  request: {item.text}")
    return "\n".join(lines)


def json_dumps(obj: Any) -> str:
    if isinstance(obj, BaseModel):
        obj = obj.model_dump()
    return json.dumps(obj, ensure_ascii=False, indent=2)


## 3. Загрузка и сохранение данных

Поддерживаются два формата входа:

### `.txt`

Один реальный запрос на строку.

### `.jsonl`

Каждая строка:

```json
{"id": "real_001", "text": "Нужно сделать ...", "metadata": {"source": "prod"}}
```


In [ ]:
def load_real_requests(path: str | Path) -> list[RealRequest]:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    rows: list[RealRequest] = []

    if path.suffix.lower() == ".jsonl":
        with path.open("r", encoding="utf-8") as f:
            for idx, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                if isinstance(obj, str):
                    rows.append(RealRequest(id=f"real_{idx:06d}", text=obj))
                else:
                    rows.append(
                        RealRequest(
                            id=str(obj.get("id", f"real_{idx:06d}")),
                            text=str(obj["text"]),
                            metadata=obj.get("metadata", {}),
                        )
                    )
    else:
        with path.open("r", encoding="utf-8") as f:
            for idx, line in enumerate(f, start=1):
                text = line.strip()
                if text:
                    rows.append(RealRequest(id=f"real_{idx:06d}", text=text))

    if not rows:
        raise ValueError("No real requests loaded")

    return rows


def save_jsonl(path: str | Path, items: list[SyntheticRequest]) -> None:
    path = Path(path)
    with path.open("w", encoding="utf-8") as f:
        for item in items:
            f.write(json.dumps(item.model_dump(), ensure_ascii=False) + "\n")


## 4. OpenAI client

Здесь используется `responses.parse` для structured outputs.


In [ ]:
class LLMClient:
    def __init__(self, model: str = DEFAULT_MODEL):
        self.client = OpenAI()
        self.model = model

    def parse(
        self,
        schema: type[BaseModel],
        system_prompt: str,
        user_prompt: str,
        retries: int = 3,
    ) -> BaseModel:
        last_error: Exception | None = None

        for attempt in range(1, retries + 1):
            try:
                response = self.client.responses.parse(
                    model=self.model,
                    input=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                    text_format=schema,
                )

                parsed = response.output_parsed
                if parsed is None:
                    raise ValueError("Model returned no parsed structured output")

                return parsed

            except Exception as exc:
                last_error = exc
                sleep_s = 1.5 * attempt
                print(f"LLM call failed on attempt {attempt}/{retries}: {exc}")
                time.sleep(sleep_s)

        raise RuntimeError(f"LLM call failed after {retries} retries: {last_error}") from last_error


## 5. Prompt 1 — построение таксономии


In [ ]:
def infer_taxonomy(
    llm: LLMClient,
    real_requests: list[RealRequest],
    max_examples: int = 80,
) -> DatasetTaxonomy:
    system = '''
Ты senior data scientist, который строит таксономию пользовательских запросов
для генерации синтетических данных.

Твоя задача:
- понять домены, интенты, стили и ограничения в реальных запросах;
- НЕ копировать персональные данные;
- выделить, что можно варьировать при генерации синтетики;
- вернуть строго структурированный результат.
'''.strip()

    user = f'''
Ниже реальные пользовательские запросы. Проанализируй их и построй таксономию.

РЕАЛЬНЫЕ ЗАПРОСЫ:
{examples_block(real_requests, max_examples=max_examples)}

Требования:
- domains: основные темы;
- intents: что пользователь хочет сделать;
- complexity_levels: уровни сложности;
- style_patterns: стиль, ошибки, язык, формат, длина;
- constraints_patterns: типы ограничений, которые часто встречаются;
- forbidden_copy_patterns: что нельзя копировать в синтетические данные.
'''.strip()

    return llm.parse(DatasetTaxonomy, system, user)  # type: ignore[return-value]


## 6. Prompt 2 — генерация synthetic requests


In [ ]:
def generate_batch(
    llm: LLMClient,
    real_examples: list[RealRequest],
    taxonomy: DatasetTaxonomy,
    batch_size: int,
    batch_index: int,
) -> SyntheticBatch:
    system = '''
Ты генератор высококачественных synthetic user requests.

Нужно генерировать запросы, которые:
- выглядят как реальные пользовательские запросы;
- вдохновлены реальными примерами, но НЕ являются копиями;
- сохраняют разнообразие интентов, доменов, стилей, сложности;
- могут содержать естественные опечатки, если они есть в реальных данных;
- не содержат реальных email, телефонов, паспортов, адресов, токенов, ключей;
- не раскрывают приватные данные из примеров;
- подходят для обучения или тестирования ассистента.

Возвращай только structured output.
'''.strip()

    user = f'''
Сгенерируй ровно {batch_size} синтетических пользовательских запросов.

ТАКСОНОМИЯ:
{json_dumps(taxonomy)}

РЕАЛЬНЫЕ ПРИМЕРЫ, КОТОРЫЕ МОЖНО ИСПОЛЬЗОВАТЬ КАК FEW-SHOT:
{examples_block(real_examples, max_examples=len(real_examples))}

Правила:
1. Каждый synthetic request должен быть новым.
2. Нельзя копировать реальные запросы дословно.
3. Нельзя заменять пару слов и выдавать это за новый запрос.
4. Нужно использовать source_example_ids — id примеров, которые вдохновили синт.
5. synthetic_id должен иметь формат synth_{batch_index:03d}_NNN.
6. Должны быть разные:
   - домены;
   - интенты;
   - длина;
   - сложность;
   - стиль формулировки.
7. Если реальные запросы с ошибками, можно сгенерировать часть запросов с естественными ошибками.
8. Если реальные запросы технические, сохраняй технический уровень.
9. Не добавляй ответ ассистента — только запрос пользователя.
'''.strip()

    return llm.parse(SyntheticBatch, system, user)  # type: ignore[return-value]


## 7. Prompt 3 — LLM-валидация


In [ ]:
def validate_batch_with_llm(
    llm: LLMClient,
    real_examples: list[RealRequest],
    taxonomy: DatasetTaxonomy,
    batch: SyntheticBatch,
) -> ValidationReport:
    system = '''
Ты строгий валидатор synthetic user requests.

Оцени каждый синтетический запрос:
- реалистичность;
- непохожесть на прямую копию;
- соответствие домену и интенту;
- отсутствие приватных данных;
- полезность для датасета;
- разнообразие;
- соответствие реальным примерам.

Возвращай structured output.
'''.strip()

    user = f'''
Провалидируй synthetic batch.

ТАКСОНОМИЯ:
{json_dumps(taxonomy)}

РЕАЛЬНЫЕ ПРИМЕРЫ:
{examples_block(real_examples, max_examples=len(real_examples))}

СИНТЕТИЧЕСКИЕ ЗАПРОСЫ:
{json_dumps(batch)}

Критерии:
- is_valid=true только если запрос можно оставить в датасете;
- score 0-100;
- score < 75 означает, что запрос лучше заменить или починить;
- issues должны быть конкретными;
- suggested_fix должен объяснять, как исправить.
'''.strip()

    return llm.parse(ValidationReport, system, user)  # type: ignore[return-value]


## 8. Prompt 4 — repair невалидных синтов


In [ ]:
def repair_batch(
    llm: LLMClient,
    real_examples: list[RealRequest],
    taxonomy: DatasetTaxonomy,
    invalid_items: list[SyntheticRequest],
    validation_report: ValidationReport,
) -> SyntheticBatch:
    system = '''
Ты исправляешь плохие synthetic user requests.

Нужно вернуть улучшенные версии:
- не копировать реальные примеры;
- исправить проблемы из validation report;
- сохранить source_example_ids;
- сделать запросы реалистичными;
- убрать приватные данные и placeholders;
- вернуть structured output.
'''.strip()

    user = f'''
Исправь все невалидные synthetic requests.

ТАКСОНОМИЯ:
{json_dumps(taxonomy)}

РЕАЛЬНЫЕ ПРИМЕРЫ:
{examples_block(real_examples, max_examples=len(real_examples))}

НЕВАЛИДНЫЕ ITEMS:
{json_dumps({"items": [x.model_dump() for x in invalid_items]})}

VALIDATION REPORT:
{json_dumps(validation_report)}

Требования:
- количество items в ответе должно совпадать с количеством invalid_items;
- synthetic_id оставь тем же;
- user_request перепиши качественно;
- variation_strategy и quality_notes обнови;
- source_example_ids должны ссылаться на реальные id.
'''.strip()

    return llm.parse(SyntheticBatch, system, user)  # type: ignore[return-value]


## 9. Локальная валидация

LLM-валидация полезна, но её нужно дополнять обычными проверками.


In [ ]:
def local_validate_item(
    item: SyntheticRequest,
    real_requests: list[RealRequest],
    accepted_items: list[SyntheticRequest],
) -> tuple[bool, list[str]]:
    issues: list[str] = []
    text = item.user_request.strip()

    if len(text) < MIN_REQUEST_LEN:
        issues.append("too_short")

    if contains_bad_placeholder(text):
        issues.append("contains_placeholder")

    if contains_pii_like_text(text):
        issues.append("contains_pii_like_pattern")

    real_ids = {r.id for r in real_requests}
    unknown_ids = [x for x in item.source_example_ids if x not in real_ids]
    if unknown_ids:
        issues.append(f"unknown_source_example_ids={unknown_ids}")

    max_real_sim = 0.0
    closest_real_id = None
    for real in real_requests:
        sim = similarity(text, real.text)
        if sim > max_real_sim:
            max_real_sim = sim
            closest_real_id = real.id

    if max_real_sim >= MAX_SIMILARITY_TO_REAL:
        issues.append(f"too_similar_to_real:{closest_real_id}:{max_real_sim:.3f}")

    max_accepted_sim = 0.0
    closest_synth_id = None
    for accepted in accepted_items:
        sim = similarity(text, accepted.user_request)
        if sim > max_accepted_sim:
            max_accepted_sim = sim
            closest_synth_id = accepted.synthetic_id

    if max_accepted_sim >= MAX_SIMILARITY_TO_ACCEPTED:
        issues.append(f"duplicate_or_too_similar_to_accepted:{closest_synth_id}:{max_accepted_sim:.3f}")

    return len(issues) == 0, issues


def merge_validation(
    batch: SyntheticBatch,
    llm_report: ValidationReport,
    real_requests: list[RealRequest],
    accepted_items: list[SyntheticRequest],
) -> tuple[list[SyntheticRequest], list[SyntheticRequest], dict[str, list[str]]]:
    report_by_id = {x.synthetic_id: x for x in llm_report.items}

    valid: list[SyntheticRequest] = []
    invalid: list[SyntheticRequest] = []
    issue_map: dict[str, list[str]] = {}

    for item in batch.items:
        issues: list[str] = []

        local_ok, local_issues = local_validate_item(item, real_requests, accepted_items + valid)
        issues.extend(local_issues)

        llm_item = report_by_id.get(item.synthetic_id)
        if llm_item is None:
            issues.append("missing_llm_validation")
        else:
            if not llm_item.is_valid:
                issues.extend([f"llm:{x}" for x in llm_item.issues])
            if llm_item.score < MIN_LLM_VALIDATION_SCORE:
                issues.append(f"llm_score_too_low:{llm_item.score}")

        if issues:
            invalid.append(item)
            issue_map[item.synthetic_id] = issues
        else:
            valid.append(item)

    return valid, invalid, issue_map


## 10. Главная функция генерации


In [ ]:
def generate_synthetics(
    input_path: str | Path,
    output_path: str | Path,
    target_count: int = 100,
    batch_size: int = 20,
    context_examples: int = 25,
    model: str = DEFAULT_MODEL,
    seed: int = 42,
    max_rounds: int = 20,
) -> list[SyntheticRequest]:
    random.seed(seed)

    real_requests = load_real_requests(input_path)
    llm = LLMClient(model=model)

    print(f"Loaded real requests: {len(real_requests)}")
    print("Inferring taxonomy...")
    taxonomy = infer_taxonomy(llm, real_requests)

    print("\nTaxonomy:")
    print(json_dumps(taxonomy))

    accepted: list[SyntheticRequest] = []
    seen_hashes: set[str] = set()

    rounds = 0

    while len(accepted) < target_count and rounds < max_rounds:
        rounds += 1

        sample_size = min(context_examples, len(real_requests))
        real_sample = random.sample(real_requests, sample_size)

        remaining = target_count - len(accepted)
        current_batch_size = min(batch_size, remaining * 2)

        print(f"\nRound {rounds}: generating batch_size={current_batch_size}...")
        batch = generate_batch(
            llm=llm,
            real_examples=real_sample,
            taxonomy=taxonomy,
            batch_size=current_batch_size,
            batch_index=rounds,
        )

        print("Validating with LLM...")
        llm_report = validate_batch_with_llm(
            llm=llm,
            real_examples=real_sample,
            taxonomy=taxonomy,
            batch=batch,
        )

        valid, invalid, issue_map = merge_validation(
            batch=batch,
            llm_report=llm_report,
            real_requests=real_requests,
            accepted_items=accepted,
        )

        print(f"Initial valid={len(valid)}, invalid={len(invalid)}")

        if invalid:
            print("Repairing invalid items...")
            repaired_batch = repair_batch(
                llm=llm,
                real_examples=real_sample,
                taxonomy=taxonomy,
                invalid_items=invalid,
                validation_report=llm_report,
            )

            repaired_report = validate_batch_with_llm(
                llm=llm,
                real_examples=real_sample,
                taxonomy=taxonomy,
                batch=repaired_batch,
            )

            repaired_valid, repaired_invalid, repaired_issues = merge_validation(
                batch=repaired_batch,
                llm_report=repaired_report,
                real_requests=real_requests,
                accepted_items=accepted + valid,
            )

            valid.extend(repaired_valid)

            if repaired_invalid:
                print(f"Still invalid after repair: {len(repaired_invalid)}")
                for bad in repaired_invalid[:5]:
                    print(f"- {bad.synthetic_id}: {repaired_issues.get(bad.synthetic_id, [])}")

        added = 0
        for item in valid:
            key = stable_hash(item.user_request)
            if key in seen_hashes:
                continue

            ok, issues = local_validate_item(item, real_requests, accepted)
            if not ok:
                continue

            seen_hashes.add(key)
            accepted.append(item)
            added += 1

            if len(accepted) >= target_count:
                break

        print(f"Added={added}. Total accepted={len(accepted)}/{target_count}")

        save_jsonl(output_path, accepted)

    if len(accepted) < target_count:
        print(
            f"Warning: generated only {len(accepted)} valid items "
            f"out of requested {target_count}. Try increasing max_rounds."
        )

    save_jsonl(output_path, accepted)
    return accepted


## 11. Пример реальных запросов

Эта ячейка создаёт маленький тестовый файл. В реальном проекте замени его на свой файл с production-запросами.


In [ ]:
sample_real_requests = [
    "Нужно написать код, который генерит синты. Он должен быть продвинутый и состоять из нескольких запросов плюс валидация.",
    "Сделай FastAPI endpoint для загрузки документов и поиска по embeddings.",
    "Нужно переписать Jinja шаблоны на React, но оставить backend на FastAPI.",
    "Сгенерируй тестовые пользовательские запросы для ML ассистента, чтобы они были похожи на реальные.",
    "Проверь код генерации линий на N×N поле для логической игры.",
    "Напиши пайплайн для генерации датасета: сначала анализ примеров, потом генерация, потом фильтрация дублей.",
    "Нужно сделать RAG поиск по загруженным документам и добавить валидацию входных файлов.",
]

sample_path = Path("real_requests_sample.txt")
sample_path.write_text("\n".join(sample_real_requests), encoding="utf-8")

sample_path


## 12. Запуск генерации

Перед запуском проверь, что `OPENAI_API_KEY` доступен в окружении.

Можно начать с маленького `target_count`, например `10`, чтобы быстро проверить пайплайн.


In [ ]:
# Пример:
# os.environ["OPENAI_API_KEY"] = "sk-..."  # лучше не хранить ключ прямо в notebook

output_path = Path("synthetic_requests.jsonl")

# Раскомментируй для запуска:
# synthetic_items = generate_synthetics(
#     input_path=sample_path,
#     output_path=output_path,
#     target_count=10,
#     batch_size=5,
#     context_examples=5,
#     model=DEFAULT_MODEL,
#     seed=42,
#     max_rounds=5,
# )

# len(synthetic_items), output_path


## 13. Просмотр результата


In [ ]:
def preview_jsonl(path: str | Path, n: int = 5) -> list[dict[str, Any]]:
    path = Path(path)
    rows = []
    if not path.exists():
        print(f"File does not exist yet: {path}")
        return rows

    with path.open("r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if idx >= n:
                break
            rows.append(json.loads(line))

    return rows


# preview_jsonl(output_path, n=5)


## 14. Что можно улучшить дальше

Идеи для production-версии:

- заменить `difflib` на embeddings similarity;
- добавить кластеризацию по интентам;
- добавить балансировку датасета по доменам и сложности;
- сохранять validation report отдельно;
- логировать все prompts и model outputs;
- добавить human-in-the-loop review;
- добавить авторазметку expected rubric / ideal answer outline;
- добавить negative examples: плохие, неоднозначные или небезопасные запросы.


---

# V2: балансировка сложности и document-ignorant вопросы

Эта версия добавляет режим генерации запросов для RAG/document QA, где **пользователь не знает документы** и не должен писать фразы вроде «в документе», «в файле», «согласно PDF», «найди в базе знаний».

Synthetic request должен звучать как обычный человеческий вопрос, например: «Могу ли я вернуть товар, если упаковку уже открыл?» или «Что делать, если я пропустил срок подачи заявки?».

При этом хороший ответ всё равно может требовать информации из документа, регламента, инструкции или базы знаний. Это отмечается как `answer_requires_document=True`.


In [ ]:
from __future__ import annotations

from collections import Counter
from typing import Literal, Any

ComplexityLevel = Literal[
    "direct_simple",
    "implicit_lookup",
    "multi_constraint",
    "procedural",
    "edge_case",
    "comparative",
    "ambiguous",
]

COMPLEXITY_DESCRIPTIONS: dict[str, str] = {
    "direct_simple": "Короткий и понятный вопрос, но без слов документ/файл/PDF.",
    "implicit_lookup": "Пользователь описывает ситуацию; нужно понять, какое правило или условие искать.",
    "multi_constraint": "В вопросе несколько условий: сроки, статус, роль, ограничения, регион или процесс.",
    "procedural": "Пользователь спрашивает, что делать и какие шаги выполнить.",
    "edge_case": "Нестандартная ситуация: исключение, конфликт правил или редкая комбинация условий.",
    "comparative": "Нужно сравнить варианты, условия, тарифы, роли, процедуры или ограничения.",
    "ambiguous": "Естественно неполный или размытый вопрос, как человек формулирует в жизни.",
}

COMPLEXITY_TARGET_MIX: dict[str, float] = {
    "direct_simple": 0.15,
    "implicit_lookup": 0.25,
    "multi_constraint": 0.18,
    "procedural": 0.15,
    "edge_case": 0.10,
    "comparative": 0.10,
    "ambiguous": 0.07,
}


def normalize_mix(mix: dict[str, float]) -> dict[str, float]:
    total = sum(max(v, 0.0) for v in mix.values())
    if total <= 0:
        raise ValueError("Complexity mix must have positive total weight")
    return {k: max(v, 0.0) / total for k, v in mix.items()}


def make_complexity_plan(
    current_items: list["SyntheticRequest"],
    batch_size: int,
    target_mix: dict[str, float] | None = None,
) -> list[str]:
    """Планирует complexity buckets для следующего batch с учетом недопредставленных классов."""
    target_mix = normalize_mix(target_mix or COMPLEXITY_TARGET_MIX)
    counts = Counter(getattr(item, "complexity_level", None) for item in current_items)
    total = max(len(current_items), 1)

    deficits = {}
    for level, target_share in target_mix.items():
        current_share = counts[level] / total
        deficits[level] = max(target_share - current_share, 0.0)

    weights = deficits if sum(deficits.values()) > 0 else target_mix
    levels = list(weights.keys())
    plan = random.choices(levels, weights=[weights[x] for x in levels], k=batch_size)
    random.shuffle(plan)
    return plan


def dataset_complexity_report(items: list["SyntheticRequest"]) -> dict[str, Any]:
    counts = Counter(item.complexity_level for item in items)
    total = len(items) or 1
    return {
        "total": len(items),
        "counts": dict(counts),
        "shares": {k: round(v / total, 4) for k, v in counts.items()},
        "target_mix": normalize_mix(COMPLEXITY_TARGET_MIX),
    }


def complexity_plan_block(plan: list[str]) -> str:
    counts = Counter(plan)
    return "\n".join(
        f"- {level}: {count} шт. — {COMPLEXITY_DESCRIPTIONS[level]}"
        for level, count in counts.items()
    )


## V2.1 Локальный фильтр document-aware формулировок

Этот фильтр отбрасывает запросы, где пользователь явно знает о документе или просит искать в источнике. Для нужного датасета такие вопросы плохие: человек должен спрашивать про свою ситуацию, а не про файл.


In [ ]:
DOCUMENT_AWARE_PATTERNS = [
    r"\bдокумент\w*\b",
    r"\bфайл\w*\b",
    r"\bpdf\b",
    r"\bпдф\b",
    r"\bотчет\w*\b",
    r"\bтаблиц\w*\b",
    r"\bпрезентаци\w*\b",
    r"\bинструкци\w*\b",
    r"\bмануал\w*\b",
    r"\bрегламент\w*\b",
    r"\bбаз[аеуы]\s+знани\w*\b",
    r"\bknowledge\s+base\b",
    r"\bdocument\w*\b",
    r"\bfile\w*\b",
    r"\battachment\w*\b",
    r"\bsource\w*\b",
    r"согласно\s+\w+",
    r"на\s+основе\s+документ\w*",
    r"найди\s+в\s+\w+",
    r"посмотри\s+в\s+\w+",
    r"что\s+написано\s+в\s+\w+",
]


def is_document_aware_request(text: str) -> bool:
    normalized = normalize_text(text)
    return any(re.search(pattern, normalized, flags=re.IGNORECASE) for pattern in DOCUMENT_AWARE_PATTERNS)


test_queries = [
    "Что написано в документе про возврат?",
    "Найди в PDF условия возврата",
    "Согласно регламенту, можно ли мне подать заявку?",
    "Могу ли я вернуть товар, если упаковку уже открыл?",
    "Что делать, если я пропустил срок подачи заявки?",
]

for q in test_queries:
    print(f"{is_document_aware_request(q):5} | {q}")


## V2.2 Новые схемы данных

После запуска этой ячейки классы `DatasetTaxonomy`, `SyntheticRequest`, `SyntheticBatch`, `ValidationItem`, `ValidationReport` переопределяются под новый режим.


In [ ]:
class DatasetTaxonomy(BaseModel):
    domains: list[str] = Field(description="Main domains/topics found in real requests.")
    intents: list[str] = Field(description="User intents found in real requests.")
    complexity_levels: list[str] = Field(description="Observed and desired complexity levels.")
    style_patterns: list[str] = Field(description="Observed language/style/noise patterns.")
    constraints_patterns: list[str] = Field(description="Common constraint types.")
    document_dependent_question_patterns: list[str] = Field(
        description="Natural question patterns that may require a document, without mentioning documents."
    )
    forbidden_document_aware_patterns: list[str] = Field(
        description="Phrases that make a request too document-aware and should be avoided."
    )
    forbidden_copy_patterns: list[str] = Field(description="Patterns that must not be copied into synthetic data.")

    @field_validator(
        "domains",
        "intents",
        "complexity_levels",
        "style_patterns",
        "constraints_patterns",
        "document_dependent_question_patterns",
        "forbidden_document_aware_patterns",
        "forbidden_copy_patterns",
    )
    @classmethod
    def non_empty_list(cls, value: list[str]) -> list[str]:
        cleaned = [x.strip() for x in value if x.strip()]
        if not cleaned:
            raise ValueError("List cannot be empty")
        return cleaned


class SyntheticRequest(BaseModel):
    synthetic_id: str = Field(description="Stable unique id, e.g. synth_001_001.")
    user_request: str = Field(description="The generated synthetic user request.")
    domain: str = Field(description="Domain/topic.")
    intent: str = Field(description="User intent.")
    complexity_level: ComplexityLevel = Field(description="Complexity bucket used for balancing.")
    expected_answer_type: str = Field(description="Expected assistant answer type.")
    answer_requires_document: bool = Field(description="True if good answer needs document/KB/rules.")
    document_dependency: str = Field(description="What hidden document knowledge is likely needed.")
    document_ignorant_formulation: str = Field(
        description="Why request sounds like it was written by a person who does not know the documents."
    )
    forbidden_document_aware_terms_present: bool = Field(
        description="True if request explicitly mentions document/file/PDF/source."
    )
    source_example_ids: list[str] = Field(description="IDs of real examples that inspired this request.")
    variation_strategy: str = Field(description="How this synthetic request differs from real examples.")
    quality_notes: str = Field(description="Why this synthetic request is realistic and useful.")

    @field_validator("user_request")
    @classmethod
    def request_must_be_non_empty(cls, value: str) -> str:
        value = value.strip()
        if len(value) < 12:
            raise ValueError("Synthetic request is too short")
        return value

    @field_validator("source_example_ids")
    @classmethod
    def source_ids_required(cls, value: list[str]) -> list[str]:
        if not value:
            raise ValueError("source_example_ids cannot be empty")
        return value


class SyntheticBatch(BaseModel):
    items: list[SyntheticRequest]


class ValidationItem(BaseModel):
    synthetic_id: str
    is_valid: bool
    score: int = Field(ge=0, le=100)
    issues: list[str]
    suggested_fix: str


class ValidationReport(BaseModel):
    items: list[ValidationItem]
    batch_score: int = Field(ge=0, le=100)
    global_issues: list[str]


## V2.3 Prompts: таксономия, генерация, валидация, repair

Ключевые изменения:

- модель явно получает правило: пользователь не знает документы;
- модель получает `complexity_plan` для текущего batch;
- валидатор обязан ставить `is_valid=false`, если вопрос содержит document-aware лексику;
- repair переписывает плохие вопросы в человеческий, неявный стиль.


In [ ]:
def infer_taxonomy(
    llm: LLMClient,
    real_requests: list[RealRequest],
    max_examples: int = 80,
) -> DatasetTaxonomy:
    system = """
Ты senior data scientist, который строит таксономию пользовательских запросов
для генерации синтетических данных для RAG/document QA.

Ключевой принцип:
пользователь НЕ знает документы и НЕ формулирует вопрос как "найди в документе".
Он задает обычный человеческий вопрос, но хороший ответ может требовать документа,
регламента, инструкции, базы знаний или внутренних правил.

Твоя задача:
- понять домены, интенты, стили и ограничения в реальных запросах;
- выделить естественные паттерны вопросов, где документ нужен неявно;
- выделить фразы, которые запрещены, потому что делают вопрос слишком document-aware;
- НЕ копировать персональные данные;
- вернуть строго структурированный результат.
""".strip()

    user = f"""
Ниже реальные пользовательские запросы. Проанализируй их и построй таксономию.

РЕАЛЬНЫЕ ЗАПРОСЫ:
{examples_block(real_requests, max_examples=max_examples)}

Используй такие уровни сложности:
{json_dumps(COMPLEXITY_DESCRIPTIONS)}

Требования:
- domains: основные темы;
- intents: что пользователь хочет сделать;
- complexity_levels: уровни сложности из списка выше;
- style_patterns: стиль, ошибки, язык, формат, длина;
- constraints_patterns: типы ограничений, которые часто встречаются;
- document_dependent_question_patterns: естественные формулировки, где ответ требует документа, но пользователь документ не упоминает;
- forbidden_document_aware_patterns: фразы, которые нужно запрещать: "в документе", "в файле", "согласно PDF", "найди в базе знаний" и подобные;
- forbidden_copy_patterns: что нельзя копировать в синтетические данные.
""".strip()

    return llm.parse(DatasetTaxonomy, system, user)  # type: ignore[return-value]


def generate_batch(
    llm: LLMClient,
    real_examples: list[RealRequest],
    taxonomy: DatasetTaxonomy,
    complexity_plan: list[str],
    batch_index: int,
) -> SyntheticBatch:
    batch_size = len(complexity_plan)

    system = """
Ты генератор высококачественных synthetic user requests для RAG/document QA.

Главное правило:
пользователь НЕ знает, какие документы есть в системе.
Он НЕ пишет "в документе", "в файле", "в PDF", "согласно регламенту", "найди в базе знаний".
Он задает обычный вопрос, который мог бы задать реальный человек.

Нужно генерировать запросы, которые:
- выглядят как реальные пользовательские вопросы;
- могут требовать информации из документов, но НЕ упоминают документы;
- вдохновлены реальными примерами, но НЕ являются копиями;
- сохраняют разнообразие интентов, доменов, стилей, сложности;
- могут содержать естественные опечатки, если они есть в реальных данных;
- не содержат реальных email, телефонов, паспортов, адресов, токенов, ключей;
- подходят для обучения или тестирования ассистента/RAG-системы.

Возвращай только structured output.
""".strip()

    user = f"""
Сгенерируй ровно {batch_size} синтетических пользовательских запросов.

ТАКСОНОМИЯ:
{json_dumps(taxonomy)}

РЕАЛЬНЫЕ ПРИМЕРЫ, КОТОРЫЕ МОЖНО ИСПОЛЬЗОВАТЬ КАК FEW-SHOT:
{examples_block(real_examples, max_examples=len(real_examples))}

ПЛАН СЛОЖНОСТИ ДЛЯ ЭТОГО BATCH:
{complexity_plan_block(complexity_plan)}

Список complexity_level для каждого item должен соответствовать этому плану:
{json_dumps(complexity_plan)}

Правила:
1. Каждый synthetic request должен быть новым.
2. Нельзя копировать реальные запросы дословно.
3. Нельзя заменять пару слов и выдавать это за новый запрос.
4. Нужно использовать source_example_ids — id примеров, которые вдохновили синт.
5. synthetic_id должен иметь формат synth_{batch_index:03d}_NNN.
6. user_request НЕ должен содержать: "документ", "файл", "PDF", "отчет", "таблица", "регламент", "база знаний", "согласно", "найди в", "посмотри в", "на основе".
7. Вопрос должен звучать так, будто человек просто описывает свою проблему или цель.
8. answer_requires_document должен быть true, если для хорошего ответа нужна информация из правил/инструкций/документов.
9. document_dependency должен объяснять, какая скрытая информация нужна.
10. document_ignorant_formulation должен объяснять, почему вопрос не выглядит как вопрос про документ.
11. forbidden_document_aware_terms_present должен быть false.
12. Не добавляй ответ ассистента — только запрос пользователя.

Примеры хорошего стиля:
- "Могу ли я вернуть товар, если упаковку уже открыл?"
- "Что делать, если я не успел подать заявку вовремя?"
- "Есть ли разница между обычной и срочной процедурой?"
- "Мне нужно оформить доступ для нового сотрудника, с чего начать?"
- "А если я уже оплатил, но передумал, деньги вернут?"
""".strip()

    return llm.parse(SyntheticBatch, system, user)  # type: ignore[return-value]


def validate_batch_with_llm(
    llm: LLMClient,
    real_examples: list[RealRequest],
    taxonomy: DatasetTaxonomy,
    batch: SyntheticBatch,
) -> ValidationReport:
    system = """
Ты строгий валидатор synthetic user requests для RAG/document QA.

Оцени каждый синтетический запрос:
- звучит ли он как реальный человеческий вопрос;
- не является ли он прямой копией реального примера;
- соответствует ли заявленному complexity_level;
- требует ли он скрытого знания из документа/инструкции/правил;
- НЕ упоминает ли он документ, файл, PDF, отчет, базу знаний или источник;
- нет ли приватных данных;
- полезен ли он для датасета.

Если user_request содержит "документ", "файл", "PDF", "согласно", "найди в", "посмотри в", "база знаний" или похожие фразы — is_valid=false.
""".strip()

    user = f"""
Провалидируй synthetic batch.

ТАКСОНОМИЯ:
{json_dumps(taxonomy)}

РЕАЛЬНЫЕ ПРИМЕРЫ:
{examples_block(real_examples, max_examples=len(real_examples))}

СИНТЕТИЧЕСКИЕ ЗАПРОСЫ:
{json_dumps(batch)}

Критерии:
- is_valid=true только если запрос можно оставить в датасете;
- score 0-100;
- score < 75 означает, что запрос лучше заменить или починить;
- issues должны быть конкретными;
- suggested_fix должен объяснять, как исправить;
- особое внимание: человек не знает документы и не должен на них ссылаться.
""".strip()

    return llm.parse(ValidationReport, system, user)  # type: ignore[return-value]


def repair_batch(
    llm: LLMClient,
    real_examples: list[RealRequest],
    taxonomy: DatasetTaxonomy,
    invalid_items: list[SyntheticRequest],
    validation_report: ValidationReport,
) -> SyntheticBatch:
    system = """
Ты исправляешь плохие synthetic user requests для RAG/document QA.

Нужно вернуть улучшенные версии:
- пользователь НЕ знает документы;
- user_request НЕ должен упоминать документ/файл/PDF/базу знаний/источник;
- вопрос должен звучать как обычная проблема человека;
- при этом хороший ответ всё равно может требовать документа, правил или инструкции;
- complexity_level нужно сохранить;
- source_example_ids нужно сохранить;
- убрать приватные данные и placeholders;
- вернуть structured output.
""".strip()

    user = f"""
Исправь все невалидные synthetic requests.

ТАКСОНОМИЯ:
{json_dumps(taxonomy)}

РЕАЛЬНЫЕ ПРИМЕРЫ:
{examples_block(real_examples, max_examples=len(real_examples))}

НЕВАЛИДНЫЕ ITEMS:
{json_dumps({"items": [x.model_dump() for x in invalid_items]})}

VALIDATION REPORT:
{json_dumps(validation_report)}

Требования:
- количество items в ответе должно совпадать с количеством invalid_items;
- synthetic_id оставь тем же;
- complexity_level оставь тем же;
- user_request перепиши в стиле человека, который не знает про документы;
- forbidden_document_aware_terms_present должен быть false;
- answer_requires_document обычно должен остаться true;
- variation_strategy и quality_notes обнови;
- source_example_ids должны ссылаться на реальные id.
""".strip()

    return llm.parse(SyntheticBatch, system, user)  # type: ignore[return-value]


## V2.4 Локальная валидация и merge с LLM-валидацией

Локальная проверка работает независимо от модели и жестко отбрасывает document-aware вопросы, даже если LLM случайно пропустил их.


In [ ]:
def local_validate_item(
    item: SyntheticRequest,
    real_requests: list[RealRequest],
    accepted_items: list[SyntheticRequest],
) -> tuple[bool, list[str]]:
    issues: list[str] = []
    text = item.user_request.strip()

    if len(text) < MIN_REQUEST_LEN:
        issues.append("too_short")
    if contains_bad_placeholder(text):
        issues.append("contains_placeholder")
    if contains_pii_like_text(text):
        issues.append("contains_pii_like_pattern")
    if is_document_aware_request(text):
        issues.append("document_aware_formulation")
    if item.forbidden_document_aware_terms_present:
        issues.append("model_marked_forbidden_document_terms_present")
    if not item.answer_requires_document:
        issues.append("answer_does_not_require_document")
    if item.complexity_level not in COMPLEXITY_DESCRIPTIONS:
        issues.append(f"unknown_complexity_level:{item.complexity_level}")

    real_ids = {r.id for r in real_requests}
    unknown_ids = [x for x in item.source_example_ids if x not in real_ids]
    if unknown_ids:
        issues.append(f"unknown_source_example_ids={unknown_ids}")

    max_real_sim, closest_real_id = 0.0, None
    for real in real_requests:
        sim = similarity(text, real.text)
        if sim > max_real_sim:
            max_real_sim, closest_real_id = sim, real.id
    if max_real_sim >= MAX_SIMILARITY_TO_REAL:
        issues.append(f"too_similar_to_real:{closest_real_id}:{max_real_sim:.3f}")

    max_accepted_sim, closest_synth_id = 0.0, None
    for accepted in accepted_items:
        sim = similarity(text, accepted.user_request)
        if sim > max_accepted_sim:
            max_accepted_sim, closest_synth_id = sim, accepted.synthetic_id
    if max_accepted_sim >= MAX_SIMILARITY_TO_ACCEPTED:
        issues.append(f"duplicate_or_too_similar_to_accepted:{closest_synth_id}:{max_accepted_sim:.3f}")

    return len(issues) == 0, issues


def merge_validation(
    batch: SyntheticBatch,
    llm_report: ValidationReport,
    real_requests: list[RealRequest],
    accepted_items: list[SyntheticRequest],
) -> tuple[list[SyntheticRequest], list[SyntheticRequest], dict[str, list[str]]]:
    report_by_id = {x.synthetic_id: x for x in llm_report.items}

    valid: list[SyntheticRequest] = []
    invalid: list[SyntheticRequest] = []
    issue_map: dict[str, list[str]] = {}

    for item in batch.items:
        issues: list[str] = []
        local_ok, local_issues = local_validate_item(item, real_requests, accepted_items + valid)
        issues.extend(local_issues)

        llm_item = report_by_id.get(item.synthetic_id)
        if llm_item is None:
            issues.append("missing_llm_validation")
        else:
            if not llm_item.is_valid:
                issues.extend([f"llm:{x}" for x in llm_item.issues])
            if llm_item.score < MIN_LLM_VALIDATION_SCORE:
                issues.append(f"llm_score_too_low:{llm_item.score}")

        if issues:
            invalid.append(item)
            issue_map[item.synthetic_id] = issues
        else:
            valid.append(item)

    return valid, invalid, issue_map


## V2.5 Главная функция генерации с балансировкой

После каждого раунда сохраняются `synthetic_requests_balanced.jsonl` и `synthetic_requests_balanced.complexity_report.json`.


In [ ]:
def save_complexity_report(path: str | Path, items: list[SyntheticRequest]) -> None:
    path = Path(path)
    report = dataset_complexity_report(items)
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")


def generate_synthetics(
    input_path: str | Path,
    output_path: str | Path,
    target_count: int = 100,
    batch_size: int = 20,
    context_examples: int = 25,
    model: str = DEFAULT_MODEL,
    seed: int = 42,
    max_rounds: int = 20,
    target_complexity_mix: dict[str, float] | None = None,
    report_path: str | Path | None = None,
) -> list[SyntheticRequest]:
    random.seed(seed)

    real_requests = load_real_requests(input_path)
    llm = LLMClient(model=model)

    output_path = Path(output_path)
    report_path = Path(report_path) if report_path else output_path.with_suffix(".complexity_report.json")

    print(f"Loaded real requests: {len(real_requests)}")
    print("Inferring taxonomy...")
    taxonomy = infer_taxonomy(llm, real_requests)
    print("\nTaxonomy:")
    print(json_dumps(taxonomy))

    accepted: list[SyntheticRequest] = []
    seen_hashes: set[str] = set()
    rounds = 0

    while len(accepted) < target_count and rounds < max_rounds:
        rounds += 1
        sample_size = min(context_examples, len(real_requests))
        real_sample = random.sample(real_requests, sample_size)

        remaining = target_count - len(accepted)
        current_batch_size = min(batch_size, remaining * 2)
        complexity_plan = make_complexity_plan(
            current_items=accepted,
            batch_size=current_batch_size,
            target_mix=target_complexity_mix or COMPLEXITY_TARGET_MIX,
        )

        print(f"\nRound {rounds}: generating batch_size={current_batch_size}...")
        print("Complexity plan:", dict(Counter(complexity_plan)))

        batch = generate_batch(
            llm=llm,
            real_examples=real_sample,
            taxonomy=taxonomy,
            complexity_plan=complexity_plan,
            batch_index=rounds,
        )

        print("Validating with LLM...")
        llm_report = validate_batch_with_llm(llm, real_sample, taxonomy, batch)

        valid, invalid, issue_map = merge_validation(batch, llm_report, real_requests, accepted)
        print(f"Initial valid={len(valid)}, invalid={len(invalid)}")

        if invalid:
            print("Repairing invalid items...")
            repaired_batch = repair_batch(llm, real_sample, taxonomy, invalid, llm_report)
            repaired_report = validate_batch_with_llm(llm, real_sample, taxonomy, repaired_batch)
            repaired_valid, repaired_invalid, repaired_issues = merge_validation(
                repaired_batch, repaired_report, real_requests, accepted + valid
            )
            valid.extend(repaired_valid)
            if repaired_invalid:
                print(f"Still invalid after repair: {len(repaired_invalid)}")
                for bad in repaired_invalid[:5]:
                    print(f"- {bad.synthetic_id}: {repaired_issues.get(bad.synthetic_id, [])}")

        added = 0
        for item in valid:
            key = stable_hash(item.user_request)
            if key in seen_hashes:
                continue
            ok, issues = local_validate_item(item, real_requests, accepted)
            if not ok:
                continue
            seen_hashes.add(key)
            accepted.append(item)
            added += 1
            if len(accepted) >= target_count:
                break

        print(f"Added={added}. Total accepted={len(accepted)}/{target_count}")
        print("Current complexity:", dataset_complexity_report(accepted)["counts"])

        save_jsonl(output_path, accepted)
        save_complexity_report(report_path, accepted)

    if len(accepted) < target_count:
        print(
            f"Warning: generated only {len(accepted)} valid items "
            f"out of requested {target_count}. Try increasing max_rounds."
        )

    save_jsonl(output_path, accepted)
    save_complexity_report(report_path, accepted)
    return accepted


## V2.6 Пример входных реальных запросов

Примеры сформулированы так, как обычно спрашивает человек: без знания документа, названия файла или внутренней структуры базы знаний.


In [ ]:
sample_real_requests_v2 = [
    "Могу ли я вернуть товар, если уже открыл упаковку?",
    "Что делать, если я пропустил срок подачи заявки?",
    "Есть ли разница между обычной и срочной процедурой?",
    "Мне нужно оформить доступ для нового сотрудника, с чего начать?",
    "А если я уже оплатил, но передумал, деньги вернут?",
    "Можно ли изменить данные после отправки формы?",
    "Что будет, если я не успею подтвердить аккаунт вовремя?",
    "Какие ограничения есть для бесплатного тарифа?",
    "Если сотрудник работает удаленно, ему всё равно нужен пропуск?",
    "Можно ли подать заявку за другого человека?",
    "Как понять, какой статус мне подходит?",
    "В каких случаях заявку могут отклонить?",
    "Что делать, если система показывает старые данные?",
    "Можно ли ускорить рассмотрение, если дедлайн уже завтра?",
]

sample_path_v2 = Path("real_requests_sample_v2.txt")
sample_path_v2.write_text("\n".join(sample_real_requests_v2), encoding="utf-8")
sample_path_v2


## V2.7 Запуск генерации

Сначала попробуй маленький `target_count`, например `10`, чтобы проверить качество и стоимость.


In [ ]:
custom_complexity_mix = {
    "direct_simple": 0.10,
    "implicit_lookup": 0.25,
    "multi_constraint": 0.20,
    "procedural": 0.15,
    "edge_case": 0.12,
    "comparative": 0.10,
    "ambiguous": 0.08,
}

output_path_v2 = Path("synthetic_requests_balanced.jsonl")
report_path_v2 = Path("synthetic_requests_balanced.complexity_report.json")

# os.environ["OPENAI_API_KEY"] = "sk-..."  # лучше не хранить ключ прямо в notebook

# Раскомментируй для запуска:
# synthetic_items_v2 = generate_synthetics(
#     input_path=sample_path_v2,
#     output_path=output_path_v2,
#     target_count=10,
#     batch_size=6,
#     context_examples=8,
#     model=DEFAULT_MODEL,
#     seed=42,
#     max_rounds=5,
#     target_complexity_mix=custom_complexity_mix,
#     report_path=report_path_v2,
# )

# len(synthetic_items_v2), output_path_v2, report_path_v2


## V2.8 Просмотр результата и отчета по сложности


In [ ]:
def preview_jsonl(path: str | Path, n: int = 5) -> list[dict[str, Any]]:
    path = Path(path)
    rows = []
    if not path.exists():
        print(f"File does not exist yet: {path}")
        return rows
    with path.open("r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if idx >= n:
                break
            rows.append(json.loads(line))
    return rows


def preview_complexity_report(path: str | Path) -> dict[str, Any]:
    path = Path(path)
    if not path.exists():
        print(f"File does not exist yet: {path}")
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


# preview_jsonl(output_path_v2, n=5)
# preview_complexity_report(report_path_v2)
